# Neural Fortran Examples: A Small Dense Network

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/simple.f90`. It builds a network with three inputs, five hidden units, and two outputs, then trains it on one fixed pair of vectors for 500 steps of stochastic gradient descent. The target is small enough that the run can be read by eye.

## The network

```{literalinclude} Fortran_Code/Section_Examples_Simple/app/simple.f90
:language: fortran
:lines: 1-18
```

`input(3)` only stores the width of the input. It has no weights. Each `dense` call that omits an activation uses the library default, which is the logistic sigmoid. The run prints that name.

A dense layer with $n_{\mathrm{in}}$ inputs and $n_{\mathrm{out}}$ outputs stores a matrix $W$ of shape $(n_{\mathrm{in}}, n_{\mathrm{out}})$ and a bias $b$ of length $n_{\mathrm{out}}$. The parameter count is

$$
n_{\mathrm{in}} n_{\mathrm{out}} + n_{\mathrm{out}}.
$$

The first dense layer is $3 \times 5 + 5 = 20$ parameters. The second is $5 \times 2 + 2 = 12$. The forward map implemented by `nf_dense_layer` is

$$
z_j = b_j + \sum_i x_i W_{ij}, \qquad a_j = \sigma(z_j), \qquad \sigma(z) = \frac{1}{1+e^{-z}}.
$$

`print_info` walks the layers and prints those shapes before any training.


## One training step

```{literalinclude} Fortran_Code/Section_Examples_Simple/app/simple.f90
:language: fortran
:lines: 20-33
```

The input and the target never change:

$$
x = (0.2,\ 0.4,\ 0.6), \qquad y = (0.123456,\ 0.246802).
$$

`forward` stores the sigmoid outputs. `backward` compares them with `y`. When the caller does not pass a loss, the network uses the quadratic loss

$$
L = \frac{1}{2}\sum_k (\hat y_k - y_k)^2, \qquad \frac{\partial L}{\partial \hat y_k} = \hat y_k - y_k.
$$

`update` then applies SGD with learning rate $\eta = 1$ and no momentum:

$$
\theta \leftarrow \theta - \eta \, g.
$$

The loop index starts at 0, so the first printed line is the prediction after one update, and the last line is after 501 updates. Every 50 steps the program prints `predict(x)` rather than the loss. By step 450 both outputs match the target at the six decimals in the format, and step 500 repeats that match.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Simple/app/simple.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Simple/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was the `simple` executable from that release build. Standard error was empty and the process exited 0. The listing is the saved standard output.

```{literalinclude} Fortran_Code/Section_Examples_Simple/simple.out
:language: text
```

## What the run did

![Outputs of the simple network](images/nf_simple.png)

The solid lines are the two printed outputs. The dashed lines are the two target components. The second output is already within 0.002 of its target at step 50. The first output takes until step 450. A sigmoid can represent both targets, because both lie in $(0, 1)$, and one sample with 32 parameters is a memorization problem.

## Conclusion

The example is the smallest training loop in the repository: construct a `network`, call `forward`, `backward`, and `update`, and print `predict`. The same three calls are the body of every later example.